# TorchTARK — Colab CPU/CUDA Smoke Validation

This notebook is a **thin orchestration layer**, maintained by GPT. All numerical work happens in the existing Codex-maintained `experiments.smoke` Python module.

**Before running:** Select **Runtime → Change runtime type → T4 GPU** (or another available CUDA GPU) when testing `DEVICE = "cuda"`. In Cell 1, enter the **actual existing Google Drive experiments folder**; no Drive path is assumed.

The repository code is checked out at the exact merge commit `27fab276f7ba047e33cc7396d6f2f01a68eb6cd7`. The notebook does not benchmark acceleration or implement the TARK solver. After the cells finish, share the Drive run directory or the saved manifest with GPT for independent validation.

In [ ]:
# Cell 1 — experiment configuration; change only these values
REPO_URL = "https://github.com/JohnnyZPiggy/TorchTARK.git"
PINNED_COMMIT = "27fab276f7ba047e33cc7396d6f2f01a68eb6cd7"
DEVICE = "cuda"    # "cuda" for the Colab GPU test; "cpu" for a CPU control run
DRIVE_RUNS_DIR = ""  # REQUIRED: existing mounted Drive folder, e.g. /content/drive/MyDrive/<your-folder>/experiments

if DEVICE not in ("cpu", "cuda"):
    raise ValueError("DEVICE must be 'cpu' or 'cuda'")
if not DRIVE_RUNS_DIR or not DRIVE_RUNS_DIR.startswith("/content/drive/"):
    raise ValueError("Set DRIVE_RUNS_DIR to an existing directory inside /content/drive/")
print("Requested device:", DEVICE)
print("Pinned Git commit:", PINNED_COMMIT)

In [ ]:
# Cell 2 — mount Drive and allocate a new, persistent run directory
from google.colab import drive
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

drive.mount("/content/drive")
runs_root = Path(DRIVE_RUNS_DIR)
if not runs_root.is_dir():
    raise FileNotFoundError(f"Drive experiments folder does not exist: {runs_root}")
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:6]
run_dir = runs_root / f"smoke_{DEVICE}_{run_id}"
run_dir.mkdir(parents=False, exist_ok=False)
print("Experiment output directory:", run_dir)

In [ ]:
# Cell 3 — clone the repository and checkout an immutable commit
import subprocess
import tempfile

repo_dir = Path(tempfile.mkdtemp(prefix="torchtark-", dir="/content"))
subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPO_URL, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", PINNED_COMMIT],
               check=True, capture_output=True, text=True)
actual_commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
).strip()
if actual_commit != PINNED_COMMIT:
    raise RuntimeError(f"Checkout commit mismatch: {actual_commit}")
if not (repo_dir / "experiments" / "smoke.py").is_file():
    raise FileNotFoundError("Expected experiments/smoke.py is missing at pinned commit")
print("Verified repository commit:", actual_commit)

In [ ]:
# Cell 4 — execute the existing Codex CLI, without duplicating numerical logic
import json
import platform
import sys
import torch

command = [
    sys.executable, "-m", "experiments.smoke",
    "--device", DEVICE, "--output-dir", str(run_dir),
]
process = subprocess.run(command, cwd=repo_dir, capture_output=True, text=True)
(run_dir / "stdout.log").write_text(process.stdout, encoding="utf-8")
(run_dir / "stderr.log").write_text(process.stderr, encoding="utf-8")
cuda_available = bool(torch.cuda.is_available())
gpu_name = None
gpu_name_error = None
if cuda_available:
    try:
        gpu_name = torch.cuda.get_device_name(0)
    except Exception as exc:
        gpu_name_error = str(exc)
run_metadata = {
    "git_commit": actual_commit,
    "requested_device": DEVICE,
    "pytorch_version": torch.__version__,
    "python_version": platform.python_version(),
    "cuda_available": cuda_available,
    "gpu_name": gpu_name,
    "gpu_name_error": gpu_name_error,
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "exit_code": process.returncode,
    "command": command,
    "status": "success" if process.returncode == 0 else "failure",
}
(run_dir / "run_metadata.json").write_text(
    json.dumps(run_metadata, indent=2) + "\n", encoding="utf-8"
)
print(process.stdout)
if process.stderr:
    print("STDERR:", process.stderr)
print("Saved artifacts:", run_dir)

In [ ]:
# Cell 5 — verify manifest, device, and outcome; never accept CPU fallback for CUDA
manifest_file = run_dir / "manifest.json"
if not manifest_file.is_file():
    raise RuntimeError(f"CLI did not produce manifest.json; see logs in {run_dir}")
manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
if process.returncode != 0 or manifest.get("status") != "success":
    raise RuntimeError(f"Smoke check failed (exit={process.returncode}): {manifest.get('error')}; logs: {run_dir}")
if manifest.get("selected_device") != DEVICE:
    raise AssertionError("Requested device mismatch")
actual_device = manifest.get("actual_tensor_device", "")
if DEVICE == "cuda" and not actual_device.startswith("cuda:"):
    raise AssertionError(f"CUDA was requested but computation ran on {actual_device}")
if DEVICE == "cpu" and actual_device != "cpu":
    raise AssertionError(f"CPU was requested but computation ran on {actual_device}")
if manifest.get("result") != [17.0, 39.0]:
    raise AssertionError("Unexpected numerical smoke result")
print("PASS — execution, result and device validated")
print("Actual device:", actual_device)
print("GPU:", run_metadata["gpu_name"])
print("Git commit:", actual_commit)
print("Drive artifacts:", run_dir)